In [1]:
from pathlib import Path
from itertools import combinations
import json
import re
import warnings

import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)
from sklearn.base import clone
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

warnings.filterwarnings("ignore")

RANDOM_STATE = 42

In [2]:
# Main ticker/model target
TICKER = "cl1s"

# Paths
TRIPLE_BARRIER_DIR = Path("../../data/features/triple_barrier")
FEATURE_SET_PATH = Path(f"../../data/features/clean_feature_set/{TICKER}_clean_feature_set.csv")

# Date settings
DATE_COL = "date"
INSTRUMENT_COL = "instrument"

# Label settings
# Change this if your triple-barrier CSV uses a different target column name.
TARGET_COL = "metalabel"

# Columns that should not be used as model features
NON_FEATURE_COLS = {
    # Identifiers / indexing
    "date",
    "instrument",

    # Target / label columns
    TARGET_COL,
    "target",
    "label",
    "y",
    "meta_label",
    "tb_label",
    "barrier_label",

    # Triple-barrier metadata / leakage-prone columns
    "num_days",
    "t1",
    "vertical_barrier_date",
    "barrier_touch_date",
    "event_end_date",
    "exit_date",
    "exit_price",
    "exit_return",
    "triple_barrier_return",
    "tb_return",
    "realised_return",
    "realized_return",
    "pnl",
    "profit",
    "barrier_touched",
    "first_barrier_touched",
    "hit_barrier",
    "pt",
    "sl",
    "pt_mult",
    "sl_mult",
    "volatility",
    "target_vol",
}

In [3]:
def extract_num_days_from_filename(path: Path) -> int | None:
    """
    Tries to extract num_days from filenames such as:
    cl1s_tb_num_days_10.csv
    triple_barrier_cl1s_10d.csv
    cl1s_pt1_sl1_10.csv
    """
    filename = path.stem.lower()

    patterns = [
        r"num_days[_\-]?(\d+)",
        r"(\d+)d",
        r"horizon[_\-]?(\d+)",
    ]

    for pattern in patterns:
        match = re.search(pattern, filename)
        if match:
            return int(match.group(1))

    return None


def load_feature_set(path: Path, ticker: str) -> pd.DataFrame:
    """
    Loads the clean feature set for a specific ticker.
    """
    if not path.exists():
        raise FileNotFoundError(f"Feature set not found: {path}")

    features = pd.read_csv(path)
    features[DATE_COL] = pd.to_datetime(features[DATE_COL])

    if INSTRUMENT_COL in features.columns:
        features[INSTRUMENT_COL] = features[INSTRUMENT_COL].str.lower()
        features = features[features[INSTRUMENT_COL] == ticker.lower()].copy()

    features = features.sort_values(DATE_COL).reset_index(drop=True)

    return features


def load_triple_barrier_files(
    triple_barrier_dir: Path,
    ticker: str,
) -> dict[str, pd.DataFrame]:
    """
    Loads all triple-barrier CSV files for the given ticker.

    Returns:
        dict where:
        key = config name / filename stem
        value = triple-barrier dataframe
    """
    if not triple_barrier_dir.exists():
        raise FileNotFoundError(f"Triple-barrier directory not found: {triple_barrier_dir}")

    csv_files = sorted(triple_barrier_dir.glob("*.csv"))

    # Keep files that mention the ticker.
    ticker_files = [
        path for path in csv_files
        if ticker.lower() in path.name.lower()
    ]

    if len(ticker_files) == 0:
        raise FileNotFoundError(
            f"No triple-barrier CSV files found for ticker '{ticker}' in {triple_barrier_dir}"
        )

    tb_data = {}

    for path in ticker_files:
        df = pd.read_csv(path)
        df[DATE_COL] = pd.to_datetime(df[DATE_COL])

        if INSTRUMENT_COL in df.columns:
            df[INSTRUMENT_COL] = df[INSTRUMENT_COL].str.lower()
            df = df[df[INSTRUMENT_COL] == ticker.lower()].copy()

        if "num_days" not in df.columns:
            extracted_num_days = extract_num_days_from_filename(path)
            if extracted_num_days is None:
                raise ValueError(
                    f"'num_days' not found in {path.name}, and could not infer it from filename."
                )
            df["num_days"] = extracted_num_days

        target_col = TARGET_COL
        df = df.rename(columns={target_col: "target"})

        df = df.sort_values(DATE_COL).reset_index(drop=True)

        tb_data[path.stem] = df

    return tb_data

In [4]:
features = load_feature_set(FEATURE_SET_PATH, TICKER)
tb_configs = load_triple_barrier_files(TRIPLE_BARRIER_DIR, TICKER)

print(f"Loaded feature set for {TICKER}: {features.shape}")
print(f"Loaded {len(tb_configs)} triple-barrier configurations:")

for config_name, tb_df in tb_configs.items():
    print(
        f"  {config_name}: shape={tb_df.shape}, "
        f"num_days={tb_df['num_days'].unique()}, "
        f"target_counts={tb_df['target'].value_counts(dropna=False).to_dict()}"
    )

display(features.head())

first_config_name = list(tb_configs.keys())[0]
display(tb_configs[first_config_name].head())

Loaded feature set for cl1s: (626, 146)
Loaded 6 triple-barrier configurations:
  cl1s_ewma_10d_tp1_5_sl1_5: shape=(324, 25), num_days=[10], target_counts={0: 174, 1: 150}
  cl1s_ewma_10d_tp2_sl2: shape=(324, 25), num_days=[10], target_counts={0: 201, 1: 123}
  cl1s_ewma_5d_tp2_sl2: shape=(329, 25), num_days=[5], target_counts={0: 265, 1: 64}
  cl1s_garman_klass_10d_tp2_sl2: shape=(324, 25), num_days=[10], target_counts={0: 179, 1: 145}
  cl1s_parkinson_10d_tp2_sl2: shape=(324, 25), num_days=[10], target_counts={0: 181, 1: 143}
  cl1s_rolling_10d_tp2_sl2: shape=(324, 25), num_days=[10], target_counts={0: 173, 1: 151}


,date,instrument,primary_signal,open,high,low,close,volume,open_interest,ret_1d,...,hmm_market_stress,hmm_market_upside_breakout,hmm_market_calm_positive,hmm_market_calm_negative,signal_x_hmm_confidence,ret_5d_x_hmm_confidence,vol_20d_x_hmm_confidence,signal_x_hmm_prob_stress,signal_x_hmm_prob_high_or_extreme_vol,signal_x_hmm_prob_positive_or_strong_upside
0,2020-01-03,cl1s,0,24.795579,25.974970,24.775315,25.553469,2.185752e+06,958523.501762,0.030566,...,0.0,1.0,0.0,0.0,0.000000,0.022015,0.009606,0.000000,0.000000,0.000000
1,2020-01-06,cl1s,0,25.820960,26.230302,25.387301,25.642633,1.786962e+06,909240.146872,0.003489,...,0.0,0.0,0.0,1.0,0.000000,0.024615,0.009457,0.000000,0.000000,0.000000
2,2020-01-07,cl1s,-1,25.496729,25.593998,25.172498,25.411618,1.437614e+06,877280.234192,-0.009009,...,0.0,0.0,0.0,1.0,-0.999224,0.016524,0.009817,-0.000372,-0.000423,-0.000404
3,2020-01-08,cl1s,0,25.468359,26.607221,23.972842,24.159275,2.974939e+06,792303.827743,-0.049282,...,1.0,0.0,0.0,0.0,0.000000,-0.023747,0.015258,0.000000,0.000000,0.000000
4,2020-01-09,cl1s,0,24.313285,24.442978,23.774251,24.139011,1.852834e+06,695693.746602,-0.000839,...,0.0,0.0,0.0,1.0,0.000000,-0.025944,0.014936,0.000000,0.000000,0.000000


,instrument,signal_column,training_end,date,close,primary_signal,vol,tp,sl,timeout_date,...,target,volatility_method,ewma_span,volatility_window,num_days,take_profit_mult,stop_loss_mult,holding_period_days,raw_touch_return,signed_touch_return
0,cl1s,cl1s,2022-01-01,2020-01-07,25.411618,-1,0.019602,24.664421,26.158815,2020-01-22,...,1,ewma,100,20,10,1.5,1.5,1,-0.049282,0.049282
1,cl1s,cl1s,2022-01-01,2020-01-22,22.984255,-1,0.019635,22.307294,23.661216,2020-02-05,...,1,ewma,100,20,10,1.5,1.5,2,-0.044942,0.044942
2,cl1s,cl1s,2022-01-01,2020-01-23,22.518412,-1,0.019641,21.854975,23.181850,2020-02-06,...,1,ewma,100,20,10,1.5,1.5,4,-0.044073,0.044073
3,cl1s,cl1s,2022-01-01,2020-01-24,21.951300,-1,0.019750,21.300989,22.601611,2020-02-07,...,1,ewma,100,20,10,1.5,1.5,6,-0.037830,0.037830
4,cl1s,cl1s,2022-01-01,2020-01-27,21.525966,-1,0.019720,20.889242,22.162689,2020-02-10,...,1,ewma,100,20,10,1.5,1.5,4,-0.029733,0.029733


In [ ]:

# Clean out-of-sample period for local evaluation
# You can change these if your group decides on another split.
TEST_START_DATE = "2022-01-01"
TEST_END_DATE = "2022-06-30"

# For meta-labeling, the cleanest setup is usually to train only where the primary model made a trade.
# The meta-model learns: should we take this +1/-1 signal?
TRAIN_ON_NONZERO_SIGNALS_ONLY = True


def get_feature_cols(df: pd.DataFrame, non_feature_cols: set[str]) -> list[str]:
    """
    Selects model feature columns from the merged dataframe.
    Keeps primary_signal as a feature unless it is explicitly in NON_FEATURE_COLS.
    """
    feature_cols = [
        col for col in df.columns
        if col not in non_feature_cols
    ]

    # Keep only numeric columns because sklearn models need numeric X.
    numeric_feature_cols = [
        col for col in feature_cols
        if pd.api.types.is_numeric_dtype(df[col])
    ]

    dropped_non_numeric = sorted(set(feature_cols) - set(numeric_feature_cols))
    if dropped_non_numeric:
        print(f"Dropped non-numeric feature columns: {dropped_non_numeric}")

    return numeric_feature_cols


def merge_features_with_tb(
    features_df: pd.DataFrame,
    tb_df: pd.DataFrame,
    ticker: str,
    target_col: str = TARGET_COL,
) -> pd.DataFrame:
    """
    Merges the clean feature set with one triple-barrier config.

    Expected:
        features_df has date, instrument, features
        tb_df has date, instrument, metalabel, num_days, etc.

    Output:
        one merged dataframe with features + target + triple-barrier metadata.
    """
    features_clean = features_df.copy()
    tb_clean = tb_df.copy()

    features_clean[DATE_COL] = pd.to_datetime(features_clean[DATE_COL])
    tb_clean[DATE_COL] = pd.to_datetime(tb_clean[DATE_COL])

    if INSTRUMENT_COL in features_clean.columns:
        features_clean[INSTRUMENT_COL] = features_clean[INSTRUMENT_COL].str.lower()
        features_clean = features_clean[features_clean[INSTRUMENT_COL] == ticker.lower()].copy()

    if INSTRUMENT_COL in tb_clean.columns:
        tb_clean[INSTRUMENT_COL] = tb_clean[INSTRUMENT_COL].str.lower()
        tb_clean = tb_clean[tb_clean[INSTRUMENT_COL] == ticker.lower()].copy()

    # If earlier loading renamed metalabel to target, rename it back/check cleanly here.
    if "target" in tb_clean.columns and target_col not in tb_clean.columns:
        tb_clean = tb_clean.rename(columns={"target": target_col})

    if target_col not in tb_clean.columns:
        raise ValueError(
            f"Target column '{target_col}' not found in triple-barrier dataframe. "
            f"Available columns: {list(tb_clean.columns)}"
        )

    merge_keys = [DATE_COL]
    if INSTRUMENT_COL in features_clean.columns and INSTRUMENT_COL in tb_clean.columns:
        merge_keys.append(INSTRUMENT_COL)

    merged = features_clean.merge(
        tb_clean,
        on=merge_keys,
        how="inner",
        suffixes=("", "_tb"),
    )

    merged = merged.sort_values(DATE_COL).reset_index(drop=True)

    return merged


def clean_model_dataframe(
    merged_df: pd.DataFrame,
    feature_cols: list[str],
    target_col: str = TARGET_COL,
    train_on_nonzero_signals_only: bool = TRAIN_ON_NONZERO_SIGNALS_ONLY,
) -> pd.DataFrame:
    """
    Cleans the merged dataframe before splitting:
    - optionally removes primary_signal == 0 rows
    - removes rows with missing target
    - replaces inf with NaN
    - drops rows with missing X or y
    """
    df = merged_df.copy()

    if train_on_nonzero_signals_only and "primary_signal" in df.columns:
        df = df[df["primary_signal"] != 0].copy()

    df = df.replace([np.inf, -np.inf], np.nan)

    required_cols = feature_cols + [target_col, DATE_COL]
    df = df.dropna(subset=required_cols).copy()

    # Make sure target is integer binary 0/1
    df[target_col] = df[target_col].astype(int)

    invalid_targets = sorted(set(df[target_col].unique()) - {0, 1})
    if invalid_targets:
        raise ValueError(
            f"Target column '{target_col}' must be binary 0/1. "
            f"Found invalid values: {invalid_targets}"
        )

    df = df.sort_values(DATE_COL).reset_index(drop=True)

    return df


def chronological_train_test_split(
    df: pd.DataFrame,
    test_start_date: str = TEST_START_DATE,
    test_end_date: str = TEST_END_DATE,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Chronological train/test split.

    Train:
        date < TEST_START_DATE

    Test:
        TEST_START_DATE <= date <= TEST_END_DATE
    """
    test_start_date = pd.to_datetime(test_start_date)
    test_end_date = pd.to_datetime(test_end_date)

    train_df = df[df[DATE_COL] < test_start_date].copy()

    test_df = df[
        (df[DATE_COL] >= test_start_date)
        & (df[DATE_COL] <= test_end_date)
    ].copy()

    return train_df, test_df


def build_xy(
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
    feature_cols: list[str],
    target_col: str = TARGET_COL,
):
    """
    Builds X/y arrays for sklearn.
    """
    X_train = train_df[feature_cols].copy()
    y_train = train_df[target_col].copy()

    X_test = test_df[feature_cols].copy()
    y_test = test_df[target_col].copy()

    return X_train, y_train, X_test, y_test

In [6]:
# =========================
# 6. Build model datasets for every triple-barrier config
# =========================

datasets_by_config = {}

for config_name, tb_df in tb_configs.items():
    print("=" * 80)
    print(f"Building dataset for triple-barrier config: {config_name}")

    merged_df = merge_features_with_tb(
        features_df=features,
        tb_df=tb_df,
        ticker=TICKER,
        target_col=TARGET_COL,
    )

    feature_cols = get_feature_cols(
        df=merged_df,
        non_feature_cols=NON_FEATURE_COLS,
    )

    model_df = clean_model_dataframe(
        merged_df=merged_df,
        feature_cols=feature_cols,
        target_col=TARGET_COL,
        train_on_nonzero_signals_only=TRAIN_ON_NONZERO_SIGNALS_ONLY,
    )

    train_df, test_df = chronological_train_test_split(
        df=model_df,
        test_start_date=TEST_START_DATE,
        test_end_date=TEST_END_DATE,
    )

    X_train, y_train, X_test, y_test = build_xy(
        train_df=train_df,
        test_df=test_df,
        feature_cols=feature_cols,
        target_col=TARGET_COL,
    )

    datasets_by_config[config_name] = {
        "merged_df": merged_df,
        "model_df": model_df,
        "train_df": train_df,
        "test_df": test_df,
        "X_train": X_train,
        "y_train": y_train,
        "X_test": X_test,
        "y_test": y_test,
        "feature_cols": feature_cols,
        "num_days": int(model_df["num_days"].mode().iloc[0]) if "num_days" in model_df.columns else None,
    }

    print(f"Merged shape:     {merged_df.shape}")
    print(f"Model shape:      {model_df.shape}")
    print(f"Train shape:      {train_df.shape}")
    print(f"Test shape:       {test_df.shape}")
    print(f"X_train shape:    {X_train.shape}")
    print(f"X_test shape:     {X_test.shape}")
    print(f"Num features:     {len(feature_cols)}")
    print(f"Train target dist:{y_train.value_counts(normalize=True).round(3).to_dict()}")
    print(f"Test target dist: {y_test.value_counts(normalize=True).round(3).to_dict()}")
    print(f"num_days:         {datasets_by_config[config_name]['num_days']}")

Building dataset for triple-barrier config: cl1s_ewma_10d_tp1_5_sl1_5
Dropped non-numeric feature columns: ['signal_column', 'timeout_date', 'touch_date', 'touched_barrier', 'training_end', 'volatility_method']
Merged shape:     (323, 169)
Model shape:      (323, 169)
Train shape:      (323, 169)
Test shape:       (0, 169)
X_train shape:    (323, 158)
X_test shape:     (0, 158)
Num features:     158
Train target dist:{0: 0.536, 1: 0.464}
Test target dist: {}
num_days:         10
Building dataset for triple-barrier config: cl1s_ewma_10d_tp2_sl2
Dropped non-numeric feature columns: ['signal_column', 'timeout_date', 'touch_date', 'touched_barrier', 'training_end', 'volatility_method']
Merged shape:     (323, 169)
Model shape:      (323, 169)
Train shape:      (323, 169)
Test shape:       (0, 169)
X_train shape:    (323, 158)
X_test shape:     (0, 158)
Num features:     158
Train target dist:{0: 0.619, 1: 0.381}
Test target dist: {}
num_days:         10
Building dataset for triple-barrier 

In [7]:
# =========================
# 7. Sanity check one config
# =========================

first_config_name = list(datasets_by_config.keys())[0]
data = datasets_by_config[first_config_name]

print(f"Using config: {first_config_name}")

display(data["train_df"].head())
display(data["test_df"].head())

print("Feature columns:")
print(data["feature_cols"])

print("\nTrain date range:")
print(data["train_df"][DATE_COL].min(), "to", data["train_df"][DATE_COL].max())

print("\nTest date range:")
print(data["test_df"][DATE_COL].min(), "to", data["test_df"][DATE_COL].max())

print("\nAny NaNs in X_train?")
print(data["X_train"].isna().sum().sort_values(ascending=False).head(10))

print("\nAny NaNs in X_test?")
print(data["X_test"].isna().sum().sort_values(ascending=False).head(10))

Using config: cl1s_ewma_10d_tp1_5_sl1_5


,date,instrument,primary_signal,open,high,low,close,volume,open_interest,ret_1d,...,metalabel,volatility_method,ewma_span,volatility_window,num_days,take_profit_mult,stop_loss_mult,holding_period_days,raw_touch_return,signed_touch_return
0,2020-01-07,cl1s,-1,25.496729,25.593998,25.172498,25.411618,1.437614e+06,8.772802e+05,-0.009009,...,1,ewma,100,20,10,1.5,1.5,1,-0.049282,0.049282
1,2020-01-22,cl1s,-1,23.599977,23.648586,22.696648,22.984255,1.530855e+06,1.194576e+06,-0.028092,...,1,ewma,100,20,10,1.5,1.5,2,-0.044942,0.044942
2,2020-01-23,cl1s,-1,22.729054,22.793867,22.186246,22.518412,1.737915e+06,1.183091e+06,-0.020268,...,1,ewma,100,20,10,1.5,1.5,4,-0.044073,0.044073
3,2020-01-24,cl1s,-1,22.558920,22.664241,21.813573,21.951300,1.447108e+06,1.203433e+06,-0.025184,...,1,ewma,100,20,10,1.5,1.5,6,-0.037830,0.037830
4,2020-01-27,cl1s,-1,21.752811,21.756861,21.116834,21.525966,1.759849e+06,1.191300e+06,-0.019376,...,1,ewma,100,20,10,1.5,1.5,4,-0.029733,0.029733


,date,instrument,primary_signal,open,high,low,close,volume,open_interest,ret_1d,...,metalabel,volatility_method,ewma_span,volatility_window,num_days,take_profit_mult,stop_loss_mult,holding_period_days,raw_touch_return,signed_touch_return


Feature columns:
['primary_signal', 'open', 'high', 'low', 'close', 'volume', 'open_interest', 'ret_1d', 'ret_5d', 'ret_20d', 'ret_63d', 'ret_126d', 'ret_252d', 'mom_sign_5d', 'mom_sign_20d', 'mom_sign_63d', 'mom_sign_126d', 'mom_sign_252d', 'ret_spread_5_20', 'ret_spread_20_63', 'price_vs_sma5', 'price_vs_sma10', 'price_vs_sma20', 'price_vs_sma50', 'price_vs_sma100', 'price_vs_sma200', 'price_vs_ema5', 'price_vs_ema10', 'price_vs_ema20', 'price_vs_ema50', 'sma5_vs_sma20', 'sma10_vs_sma50', 'sma20_vs_sma50', 'sma50_vs_sma100', 'sma50_vs_sma200', 'sma100_vs_sma200', 'ema5_vs_ema20', 'ema12_vs_ema26', 'ema20_vs_ema50', 'ema20_vs_ema100', 'sma20_slope', 'sma50_slope', 'sma100_slope', 'ema20_slope', 'ema50_slope', 'macd_line', 'macd_signal', 'macd_hist', 'macd_hist_chg', 'macd_zscore', 'vol_5d', 'vol_10d', 'vol_20d', 'vol_63d', 'vol_126d', 'vol_252d', 'ewma_vol_5d', 'ewma_vol_10d', 'ewma_vol_20d', 'ewma_vol_63d', 'vol_ratio_5_20d', 'vol_ratio_20_63d', 'vol_ratio_63_126d', 'park_vol_5d', 'p